In [1]:
!nvidia-smi -L 2>/dev/null || echo "No GPU (fine for this session)"
import sys; print("Python:", sys.version.split()[0])

GPU 0: Tesla T4 (UUID: GPU-0b9ae838-54e3-e23a-60d3-503e4e5883fd)
Python: 3.13.15


In [2]:
from pathlib import Path

WORK = Path("/content/drive/MyDrive/MiFO")
for sub in ["data/raw/fakenewsnet", "data/raw/liar", "data/processed", "notebooks"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print("Workspace ready:", WORK)
print("Contents:", [p.name for p in WORK.iterdir()])

Workspace ready: /content/drive/MyDrive/MiFO
Contents: ['notebooks', 'data']


In [3]:
import hashlib, urllib.request

RAW = WORK / "data/raw/fakenewsnet"
BASE = "https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset"
EXPECTED = {  # first 16 hex of SHA256, from provenance record
    "politifact_fake.csv": "abe7fe7aad801b1e",
    "politifact_real.csv": "2500f86a7addca0f",
    "gossipcop_fake.csv":  "c6932bffadb1230b",
    "gossipcop_real.csv":  "d721e9a8b7e660da",
}

for name, want in EXPECTED.items():
    dest = RAW / name
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}", dest)
    got = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
    print(f"{'PASS' if got == want else 'FAIL'}  {name}  {got}")

PASS  politifact_fake.csv  abe7fe7aad801b1e
PASS  politifact_real.csv  2500f86a7addca0f
PASS  gossipcop_fake.csv  c6932bffadb1230b
PASS  gossipcop_real.csv  d721e9a8b7e660da


In [4]:
import hashlib, zipfile

url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
zpath = WORK / "data/raw/liar_dataset.zip"
if not zpath.exists():
    urllib.request.urlretrieve(url, zpath)

sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
print("LIAR zip SHA256:", sha)          # <-- paste this to me
print("Size (MB):", round(zpath.stat().st_size / 1e6, 1))

with zipfile.ZipFile(zpath) as z:
    z.extractall(WORK / "data/raw/liar")
    print("Extracted:", z.namelist())

LIAR zip SHA256: 611c1addad919743dde15822b87a60bfb760d8f85597f25289e34621800654c7
Size (MB): 1.0
Extracted: ['README', 'test.tsv', 'train.tsv', 'valid.tsv']


In [5]:
import pandas as pd

COLS = ["json_id","statement_id","label","statement","subjects","speaker","job",
        "state","party","barely_true","false","half_true","mostly_true","pants_fire","context"]

liar = {}
for split in ["train", "valid", "test"]:
    df = pd.read_csv(WORK / f"data/raw/liar/{split}.tsv", sep="\t",
                     names=COLS, on_bad_lines="warn", quoting=3)
    liar[split] = df
    print(f"{split}: {df.shape[0]} rows x {df.shape[1]} cols")

print("\nLabel distribution (train):")
print(liar["train"]["label"].value_counts())
print("\nSample statement:", liar["train"]["statement"].iloc[0][:120])
print("Sample label:", liar["train"]["label"].iloc[0])

train: 10269 rows x 15 cols
valid: 1284 rows x 15 cols
test: 1283 rows x 15 cols

Label distribution (train):
label
On changing the rules for filibusters on presidential nominees                                                                                                                                                 3
During Sherrod Browns past decade as a D.C. politician, more than one out of every four jobs that has left America, left from Ohio. ... Sherrod Brown will own these horrendous Ohio job numbers next year.    2
"Obama says Iran is a 'tiny' country, 'doesn't pose a serious threat.'"                                                                                                                                        2
On support for the Export-Import Bank                                                                                                                                                                          2
Says Mitt Romney flip-flopped on abortion.      

In [6]:
import pandas as pd

COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]  # 14, in true order

liar = {}
for split in ["train", "valid", "test"]:
    df = pd.read_csv(WORK / f"data/raw/liar/{split}.tsv", sep="\t",
                     names=COLS, quoting=3)
    liar[split] = df
    print(f"{split}: {df.shape[0]} rows x {df.shape[1]} cols")

print("\nLabel distribution (train):")
print(liar["train"]["label"].value_counts())
print("\nSample statement:", liar["train"]["statement"].iloc[0][:100])
print("Sample label:", liar["train"]["label"].iloc[0])
print("Sample context:", str(liar["train"]["context"].iloc[0])[:100])

train: 10269 rows x 14 cols
valid: 1284 rows x 14 cols
test: 1283 rows x 14 cols

Label distribution (train):
label
half-true      2123
false          1998
mostly-true    1966
true           1683
barely-true    1657
pants-fire      842
Name: count, dtype: int64

Sample statement: Says the Annies List political group supports third-trimester abortions on demand.
Sample label: false
Sample context: a mailer


In [7]:
import pandas as pd

RAW = WORK / "data/raw/fakenewsnet"

frames = []
for group in ["politifact", "gossipcop"]:
    for label in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{group}_{label}.csv")
        df["source_group"] = group
        df["label_name"] = label
        df["label"] = 1 if label == "fake" else 0
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

# tweet_count: IDs are TAB-separated; NaN / empty / trailing tabs all -> correct count
fn["tweet_count"] = (fn["tweet_ids"].fillna("").astype(str)
                     .str.split("\t").map(lambda p: len([x for x in p if x.strip()])))

print("Total articles:", len(fn), "\n")
print(fn.groupby(["source_group", "label_name"]).agg(
    articles=("id", "size"),
    zero_tweets=("tweet_count", lambda s: int((s == 0).sum())),
    pct_zero_tweets=("tweet_count", lambda s: round(100 * (s == 0).mean(), 1)),
    median_tweets=("tweet_count", "median"),
).to_string())

Total articles: 23196 

                         articles  zero_tweets  pct_zero_tweets  median_tweets
source_group label_name                                                       
gossipcop    fake            5323          188              3.5           12.0
             real           16817         1058              6.3           45.0
politifact   fake             432           40              9.3           79.0
             real             624          215             34.5            8.0


In [11]:
from scipy.stats import mannwhitneyu

rows = []
for group in ["politifact", "gossipcop"]:
    for lab in ["fake", "real"]:
        s = fn[(fn.source_group == group) & (fn.label_name == lab)]["tweet_count"]
        rows.append({
            "group": group, "label": lab, "n": len(s),
            "mean": round(s.mean(), 1), "median": s.median(),
            "p25": s.quantile(.25), "p75": s.quantile(.75),
            "p90": s.quantile(.90), "p99": s.quantile(.99), "max": s.max(),
        })
dist = pd.DataFrame(rows)
print(dist.to_string(index=False), "\n")

for group in ["politifact", "gossipcop"]:
    a = fn[(fn.source_group == group) & (fn.label_name == "fake")]["tweet_count"]
    b = fn[(fn.source_group == group) & (fn.label_name == "real")]["tweet_count"]
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    r = round(2 * u / (len(a) * len(b)) - 1, 3)  # rank-biserial: +1 = fake higher
    verdict = "FAKE spreads more" if r > 0 else "REAL spreads more"
    print(f"{group:12s} U-test: p={p:.2e}  rank-biserial={r:+.3f}  -> {verdict}")

WORK = Path("C:/Users/anush/MiFO")
out = WORK / "data/"
out.mkdir(parents=True, exist_ok=True)
dist.to_csv(out / "tweet_distributions.csv", index=False)
print("\nSaved ->", out / "tweet_distributions.csv")

     group label     n  mean  median  p25   p75    p90      p99   max
politifact  fake   432 382.8    79.0 12.0 291.5  762.7  4755.89 29060
politifact  real   624 670.1     8.0  0.0 167.0 1540.2 12504.79 27377
 gossipcop  fake  5323 112.4    12.0  5.0  42.0  303.0  1039.00  2568
 gossipcop  real 16817  52.4    45.0 18.0  66.0   91.0   310.00  2082 

politifact   U-test: p=7.70e-14  rank-biserial=+0.268  -> FAKE spreads more
gossipcop    U-test: p=2.32e-193  rank-biserial=-0.269  -> REAL spreads more

Saved -> C:/Users/anush/MiFO/data/tweet_distributions.csv


In [12]:
from urllib.parse import urlparse

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan":
        return ""
    if not u.startswith(("http://", "https://")):
        u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net

fn["domain"] = fn["news_url"].map(domain_of)

for (g, l), sub in fn.groupby(["source_group", "label_name"]):
    top = sub["domain"].replace("", "(no domain)").value_counts().head(8)
    print(f"\n-- {g}/{l}: top domains --")
    print(top.to_string())

for g in ["politifact", "gossipcop"]:
    fake_d  = set(fn[(fn.source_group == g) & (fn.label_name == "fake")]["domain"]) - {""}
    real_d  = set(fn[(fn.source_group == g) & (fn.label_name == "real")]["domain"]) - {""}
    shared  = fake_d & real_d
    n_shared = fn[(fn.source_group == g) & fn.domain.isin(shared)].shape[0]
    n_total  = fn[fn.source_group == g].shape[0]
    print(f"\n{g}: {len(fake_d)} fake domains, {len(real_d)} real domains, "
          f"{len(shared)} shared (Jaccard {len(shared)/len(fake_d | real_d):.3f}); "
          f"{n_shared}/{n_total} articles ({100*n_shared/n_total:.1f}%) come from shared domains")


-- gossipcop/fake: top domains --
domain
hollywoodlife.com    460
(no domain)          256
people.com           216
dailymail.co.uk      194
radaronline.com      174
eonline.com          154
usmagazine.com       147
imdb.com             145

-- gossipcop/real: top domains --
domain
people.com               1570
dailymail.co.uk           770
en.wikipedia.org          618
etonline.com              585
longroom.com              562
usmagazine.com            562
usatoday.com              300
hollywoodreporter.com     298

-- politifact/fake: top domains --
domain
web.archive.org             69
yournewswire.com            15
facebook.com                 6
thegatewaypundit.com         5
worldnewsdailyreport.com     4
thehill.com                  4
react365.com                 4
washingtonpost.com           4

-- politifact/real: top domains --
domain
web.archive.org    127
(no domain)         57
youtube.com         47
politifact.com      24
abcnews.go.com      24
nytimes.com         22
cq.c

In [15]:
import numpy as np

fn_pos = fn[fn["domain"] != ""].copy()
print(f"Articles with a parseable domain: {len(fn_pos)} / {len(fn)}")

global_majority = int(fn["label"].mean() >= 0.5)  # 0 = real (real dominates)
print(f"Majority-class baseline (always predict real): "
      f"{max(fn['label'].mean(), 1 - fn['label'].mean()):.3f}\n")

stats = fn_pos.groupby("domain")["label"].agg(n="size", n_fake="sum")
stats["n_real"] = stats["n"] - stats["n_fake"]
fn_pos = fn_pos.merge(stats, left_on="domain", right_index=True)

def accuracy(sub):
    y = sub["label"].to_numpy()
    nf, nr = sub["n_fake"].to_numpy(), sub["n_real"].to_numpy()
    # leave-one-out prediction
    pred = np.where(nf - (y == 1) > nr - (y == 0), 1,
           np.where(nf - (y == 1) < nr - (y == 0), 0, global_majority))
    return (pred == y).mean(), len(y)

for name, sub in [("ALL", fn_pos),
                  ("politifact", fn_pos[fn_pos.source_group == "politifact"]),
                  ("gossipcop",  fn_pos[fn_pos.source_group == "gossipcop"])]:
    acc, n = accuracy(sub)
    print(f"Domain-prior LOO accuracy [{name:10s}]: {acc:.3f}  (n={n})")

# ambiguous domains only: where the domain publishes both labels
amb = fn_pos[(fn_pos.n_fake > 0) & (fn_pos.n_real > 0)]
acc_amb, n_amb = accuracy(amb)
print(f"\nOn BOTH-label domains only: {acc_amb:.3f}  (n={n_amb}) "
      f"— this is the share of data where content actually has to do the work")

Articles with a parseable domain: 22866 / 23196
Majority-class baseline (always predict real): 0.752

Domain-prior LOO accuracy [ALL       ]: 0.834  (n=22866)
Domain-prior LOO accuracy [politifact]: 0.679  (n=995)
Domain-prior LOO accuracy [gossipcop ]: 0.841  (n=21871)

On BOTH-label domains only: 0.806  (n=17296) — this is the share of data where content actually has to do the work


In [18]:
import numpy as np

d = fn_pos.groupby("domain")["label"].agg(n="size", n_fake="sum")
d["fake_share"] = d["n_fake"] / d["n"]

def bucket(fs):
    if fs >= 0.9: return "clean_fake"
    if fs <= 0.1: return "clean_real"
    return "MIXED"
d["bucket"] = d["fake_share"].map(bucket)

print(d.groupby("bucket").agg(domains=("n", "size"),
                              articles=("n", "sum")).to_string(), "\n")

fn_pos = fn_pos.merge(d[["bucket"]], left_on="domain", right_index=True)

for b in ["clean_fake", "clean_real", "MIXED"]:
    sub = fn_pos[fn_pos["bucket"] == b]
    y = sub["label"].to_numpy()
    nf, nr = sub["n_fake"].to_numpy(), sub["n_real"].to_numpy()
    pred = np.where(nf - (y == 1) > nr - (y == 0), 1,
           np.where(nf - (y == 1) < nr - (y == 0), 0, global_majority))
    print(f"{b:10s}: LOO acc = {(pred == y).mean():.3f}  (n={len(sub)})")

minority_mass = int(d[["n_fake", "n_real"]].min(axis=1).sum())
print(f"\nMinority-label articles (domain-prior's irreducible errors): {minority_mass}")
print(f"Actual domain-prior errors: {int(0.166 * len(fn_pos))} (approx) — compare")

            domains  articles
bucket                       
MIXED           373     14098
clean_fake      546      1557
clean_real     1510      7211 



KeyError: 'bucket'

In [19]:
# missing URLs
missing = fn[fn["domain"] == ""]
print(f"No parseable URL: {len(missing)} articles "
      f"({100*len(missing)/len(fn):.1f}%)")

# archive links
arch = fn[fn["domain"] == "web.archive.org"]
print(f"web.archive.org links (dead originals): {len(arch)}")

# duplicates
dup_url = fn[fn["news_url"].notna() & (fn["news_url"] != "")]
n_dup_url = int(dup_url.duplicated(subset="news_url").sum())
n_dup_title = int(fn[fn["title"].notna()].duplicated(subset="title").sum())
print(f"Duplicate URLs: {n_dup_url}, duplicate exact titles: {n_dup_title}")

# breadth
print(f"Unique domains: {fn[fn['domain'] != '']['domain'].nunique()}")
print(f"Unique TLDs: {fn[fn['domain'] != '']['domain'].str.split('.').str[-1].nunique()}")

tlds = (fn.loc[fn["domain"] != "", "domain"].str.split(".").str[-1]
        .value_counts().head(10))
print("\nTop TLDs:"); print(tlds.to_string())

out = WORK / "data/processed/step2"; out.mkdir(parents=True, exist_ok=True)
d.to_csv(out / "domain_buckets.csv")
print("\nSaved ->", out / "domain_buckets.csv")

No parseable URL: 330 articles (1.4%)
web.archive.org links (dead originals): 204
Duplicate URLs: 1208, duplicate exact titles: 1472
Unique domains: 2429
Unique TLDs: 75

Top TLDs:
domain
com     18735
uk       1694
org      1289
au        337
net       109
gov       100
news       72
ca         69
co         61
ie         55

Saved -> C:/Users/anush/MiFO/data/processed/step2/domain_buckets.csv


In [22]:
import sys, hashlib, zipfile, urllib.request
from pathlib import Path

print("platform:", sys.platform, "| cwd:", Path.cwd())
print("C:/Users/anush/MiFO        exists:", Path("C:/Users/anush/MiFO").exists())
print("/mnt/c/Users/anush/MiFO    exists:", Path("/mnt/c/Users/anush/MiFO").exists())
print("literal './C:' dir in cwd  exists:", (Path.cwd() / "C:").exists())

# auto-pick the base that actually contains our data
WORK = None
for cand in [Path("C:/Users/anush/MiFO"), Path("/mnt/c/Users/anush/MiFO")]:
    if (cand / "data/raw/fakenewsnet/politifact_fake.csv").exists():
        WORK = cand
        break

if WORK is None:
    print("\n>> Could not auto-locate FakeNewsNet — paste the diagnostic lines above back to me.")
else:
    print("\nWORK =", WORK)
    raw = WORK / "data/raw"
    raw.mkdir(parents=True, exist_ok=True)          # the fix for my bug
    zpath = raw / "liar_dataset.zip"
    if not zpath.exists():
        print("downloading LIAR...")
        urllib.request.urlretrieve("https://www.cs.ucsb.edu/~william/data/liar_dataset.zip", zpath)
    sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
    assert sha.startswith("611c1addad919743"), f"CHECKSUM MISMATCH: {sha}"
    print("LIAR zip checksum OK:", sha[:16])

    liar_dir = raw / "liar"
    liar_dir.mkdir(exist_ok=True)
    if not (liar_dir / "train.tsv").exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall(liar_dir)
    print("Files:", sorted(p.name for p in liar_dir.iterdir()))

platform: linux | cwd: /content
C:/Users/anush/MiFO        exists: True
/mnt/c/Users/anush/MiFO    exists: False
literal './C:' dir in cwd  exists: True

>> Could not auto-locate FakeNewsNet — paste the diagnostic lines above back to me.


In [24]:
#Cell C3
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

WORK = Path("/content/drive/MyDrive/MiFO")
COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]

liar = {s: pd.read_csv(WORK / f"data/raw/liar/{s}.tsv", sep="\t",
                       names=COLS, quoting=3) for s in ["train","valid","test"]}

# Train TF-IDF + Logistic Regression
pipe = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                     LogisticRegression(max_iter=2000))
pipe.fit(liar["train"]["statement"], liar["train"]["label"])
pred = pipe.predict(liar["test"]["statement"])

maj = liar["train"]["label"].value_counts(normalize=True).iloc[0]
acc = accuracy_score(liar["test"]["label"], pred)
f1m = f1_score(liar["test"]["label"], pred, average="macro")

print(f"Majority Class Baseline: {maj:.3f}")
print(f"TF-IDF + Logistic Regression Accuracy: {acc:.3f}")
print(f"Macro-F1 Score: {f1m:.3f}\n")
print(classification_report(liar["test"]["label"], pred, digits=2))

out = WORK / "data/processed/step3_liar"; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"y": liar["test"]["label"], "pred": pred}).to_csv(out / "lr_baseline_preds.csv", index=False)
print("Saved predictions to ->", out / "lr_baseline_preds.csv")

Majority Class Baseline: 0.207
TF-IDF + Logistic Regression Accuracy: 0.252
Macro-F1 Score: 0.219

              precision    recall  f1-score   support

 barely-true       0.23      0.14      0.18       214
       false       0.31      0.40      0.35       250
   half-true       0.22      0.29      0.25       267
 mostly-true       0.23      0.27      0.25       249
  pants-fire       0.33      0.03      0.06        92
        true       0.25      0.21      0.23       211

    accuracy                           0.25      1283
   macro avg       0.26      0.22      0.22      1283
weighted avg       0.25      0.25      0.24      1283

Saved predictions to -> /content/drive/MyDrive/MiFO/data/processed/step3_liar/lr_baseline_preds.csv


In [25]:
# C4 — content vs context vs combined (LIAR)
import pandas as pd, numpy as np
from pathlib import Path
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

WORK = Path("/content/drive/MyDrive/MiFO")
COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]
liar = {s: pd.read_csv(WORK / f"data/raw/liar/{s}.tsv", sep="\t",
                       names=COLS, quoting=3) for s in ["train","valid","test"]}
HIST = ["barely_true","false","half_true","mostly_true","pants_fire"]
LABEL2COL = {"barely-true":"barely_true","false":"false","half-true":"half_true",
             "mostly-true":"mostly_true","pants-fire":"pants_fire"}  # 'true' has no column

def hist_features(df, loo):
    H = df[HIST].apply(pd.to_numeric, errors="coerce").fillna(0).copy()
    if loo:  # remove this statement's own contribution to its speaker's tally
        for i, lab in enumerate(df["label"]):
            col = LABEL2COL.get(lab)
            if col and H.iloc[i][col] > 0:
                H.at[H.index[i], col] -= 1
    return np.log1p(H).to_numpy(dtype=float)

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)
Xtr_t = tfidf.fit_transform(liar["train"]["statement"])
Xte_t = tfidf.transform(liar["test"]["statement"])
ytr, yte = liar["train"]["label"], liar["test"]["label"]

def run(name, Xtr, Xte):
    m = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
    p = m.predict(Xte)
    print(f"{name:28s} acc={accuracy_score(yte, p):.3f}  macroF1={f1_score(yte, p, average='macro'):.3f}")
    return p

print("--- reference ---")
run("text only (TF-IDF)", Xtr_t, Xte_t)

print("--- context only ---")
run("history only (raw)", csr_matrix(hist_features(liar["train"], False)),
                             csr_matrix(hist_features(liar["test"],  False)))
run("history only (LOO)", csr_matrix(hist_features(liar["train"], True)),
                             csr_matrix(hist_features(liar["test"],  True)))

print("--- combined ---")
run("text + history (raw)", hstack([Xtr_t, csr_matrix(hist_features(liar["train"], False))]),
                            hstack([Xte_t, csr_matrix(hist_features(liar["test"],  False))]))
p = run("text + history (LOO)", hstack([Xtr_t, csr_matrix(hist_features(liar["train"], True))]),
                                hstack([Xte_t, csr_matrix(hist_features(liar["test"],  True))]))

pd.DataFrame({"y": yte, "pred": p}).to_csv(
    WORK / "data/processed/step3_liar/combined_preds.csv", index=False)
print("\nsaved -> data/processed/step3_liar/combined_preds.csv")

--- reference ---
text only (TF-IDF)           acc=0.252  macroF1=0.219
--- context only ---
history only (raw)           acc=0.449  macroF1=0.443
history only (LOO)           acc=0.233  macroF1=0.177
--- combined ---
text + history (raw)         acc=0.443  macroF1=0.442
text + history (LOO)         acc=0.272  macroF1=0.263

saved -> data/processed/step3_liar/combined_preds.csv
